### The begining of attention
in the following blocks of code we simply just play around with the idea of attention scores.
Attention scores are dot products of one token embedgging (a query) and the rest of the token embeddings.

this dot product is a reflection of how similiar words are to eachother. such as tea and coffee.

In [117]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], 
[0.55, 0.87, 0.66], 
[0.57, 0.85, 0.64], 
[0.22, 0.58, 0.33], 
[0.77, 0.25, 0.10], 
[0.05, 0.80, 0.55]] 
)

In [118]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544000625610, 1.4950001239777, 1.4754000902176, 0.8434000015259,
        0.7070000171661, 1.0865000486374])


In [119]:
normalized_attn_weights = torch.softmax(attn_scores_2, dim=0)
print(normalized_attn_weights)

tensor([0.1385475695133, 0.2378913015127, 0.2332740277052, 0.1239915862679,
        0.1081818640232, 0.1581136137247])


In [120]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 +=normalized_attn_weights[i]*x_i
print(context_vec_2)

tensor([0.4418657124043, 0.6514819860458, 0.5683088302612])


In [121]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9994999766350, 0.9544000029564, 0.9422000050545, 0.4753000140190,
         0.4575999677181, 0.6310000419617],
        [0.9544000029564, 1.4950001239777, 1.4754000902176, 0.8434000015259,
         0.7070000171661, 1.0865000486374],
        [0.9422000050545, 1.4754000902176, 1.4570000171661, 0.8295999765396,
         0.7153999805450, 1.0605000257492],
        [0.4753000140190, 0.8434000015259, 0.8295999765396, 0.4936999678612,
         0.3473999798298, 0.6564999818802],
        [0.4575999677181, 0.7070000171661, 0.7153999805450, 0.3473999798298,
         0.6653999686241, 0.2935000061989],
        [0.6310000419617, 1.0865000486374, 1.0605000257492, 0.6564999818802,
         0.2935000061989, 0.9450000524521]])


In [122]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098347693682, 0.2005814611912, 0.1981492340565, 0.1242282241583,
         0.1220487207174, 0.1451576650143],
        [0.1385475695133, 0.2378913015127, 0.2332740277052, 0.1239915862679,
         0.1081818640232, 0.1581136137247],
        [0.1390075981617, 0.2369214594364, 0.2326019555330, 0.1242044046521,
         0.1108002066612, 0.1564644277096],
        [0.1435268819332, 0.2073944211006, 0.2045520246029, 0.1461922228336,
         0.1262952536345, 0.1720391958952],
        [0.1526108533144, 0.1958386749029, 0.1974906474352, 0.1366866827011,
         0.1878589093685, 0.1295142918825],
        [0.1384711712599, 0.2183637171984, 0.2127594351768, 0.1420475691557,
         0.0988063737750, 0.1895517557859]])


In [123]:
attn_weights[0].sum(), attn_weights.sum(dim=-1)

(tensor(1.0000001192093),
 tensor([1.0000001192093, 1.0000000000000, 1.0000001192093, 1.0000000000000,
         1.0000000000000, 1.0000000000000]))

In [124]:
context_vectors = attn_weights @ inputs
print(context_vectors)

tensor([[0.4420594274998, 0.5930986404419, 0.5789891481400],
        [0.4418657422066, 0.6514819860458, 0.5683088302612],
        [0.4431275427341, 0.6495946049690, 0.5670731067657],
        [0.4303897619247, 0.6298280954361, 0.5510270595551],
        [0.4671017527580, 0.5909928083420, 0.5265965461731],
        [0.4177244901657, 0.6503232121468, 0.5645352602005]])


In [125]:
context_vec_2, context_vectors[1], torch.equal(context_vec_2,context_vectors) ,torch.allclose(context_vec_2, context_vectors[1])

(tensor([0.4418657124043, 0.6514819860458, 0.5683088302612]),
 tensor([0.4418657422066, 0.6514819860458, 0.5683088302612]),
 False,
 True)

here i found that two context vectors i created werent exactly the same so i printed out to see why!

In [126]:
torch.set_printoptions(precision=13)
print(context_vec_2)
print(context_vectors[1])

tensor([0.4418657124043, 0.6514819860458, 0.5683088302612])
tensor([0.4418657422066, 0.6514819860458, 0.5683088302612])


### This is the begining of real attention implementation. Still a simple approach first till we get to multilayer head attention architecture

In [127]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [128]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [129]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306369423866, 1.4550582170486])


In [130]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)



keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [131]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8523844480515)


### Now a question i've had was why do we even have a weight matrix for query. Why dont we just use the embeddings tokens. 

The conclusion I reached was by using a weight matrix we are giving the model freedom to further tune the activation of important words rather than it being a fixed similiarty output. It will be further tuned to fit the context. 

What all of that means is that we are giving the model more room to model complex relationships with the similarities of tokens!

the same concept applies to the **key** when we are looking at the query and needing to feed it some word. the key provides some sort of direction to what that word that should be fed is!



In [132]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8523844480515)


In [133]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2704830169678, 1.8523844480515, 1.8111065626144, 1.0795172452927,
        0.5577305555344, 1.5439705848694])


#### We scale the weights to ensure that we maintain the information each number holds. Softmaxing before scaling would cause big numbers to lose their meaning!

In [134]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500195115805, 0.2263837903738, 0.2198716253042, 0.1310700774193,
        0.0906289070845, 0.1820261180401])


In [135]:
context_vec_2_2= attn_weights_2 @ values
print(context_vec_2_2)

tensor([0.3061002194881, 0.8210303783417])


### Making a class for attention!

In [136]:
import torch.nn as nn
class SelfAttentionWithLinearLayers(nn.Module):
    def __init__(self, d_in, d_out,qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [137]:
torch.manual_seed(789)
FirstAttention = SelfAttentionWithLinearLayers(d_in, d_out)
print(FirstAttention(inputs))

tensor([[-0.0738902539015,  0.0712899193168],
        [-0.0748107209802,  0.0703093186021],
        [-0.0748561918736,  0.0702416673303],
        [-0.0760016292334,  0.0684501156211],
        [-0.0763276070356,  0.0679428279400],
        [-0.0754442811012,  0.0693049430847]], grad_fn=<MmBackward0>)


In [138]:
FirstAttention.W_key.weight

Parameter containing:
tensor([[ 0.4058058261871, -0.4704205393791,  0.2368052005768],
        [ 0.2133607268333, -0.2600506544113, -0.5105429887772]],
       requires_grad=True)

#### a task from the attention chapter was to output the same output using the nn.linear class using selfattention_v1!



In [150]:
import torch
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    def __init__(self, first_attention):
        super().__init__()

        self.W_query = first_attention.W_query.weight
        self.W_key = first_attention.W_key.weight
        self.W_value = first_attention.W_value.weight

    def forward(self, x):
        keys = x @ self.W_key.T
        queries = x @ self.W_query.T
        values = x @ self.W_value.T

        attn_scores = queries @ keys.transpose(-2, -1)

        attn_weights = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5),
            dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

In [151]:
# Does it give the same result?!
torch.manual_seed(789)
AttentionCheck = SelfAttention_v1(FirstAttention)
print(AttentionCheck(inputs))

tensor([[-0.0738902539015,  0.0712899193168],
        [-0.0748107209802,  0.0703093186021],
        [-0.0748561918736,  0.0702416673303],
        [-0.0760016292334,  0.0684501156211],
        [-0.0763276070356,  0.0679428279400],
        [-0.0754442811012,  0.0693049430847]], grad_fn=<MmBackward0>)


### Success!!! :)